# Reproducible Regression Quickstart

This self-contained notebook demonstrates a deterministic regression workflow using synthetic, non-personal data. It covers data generation, validation, splitting, modeling, evaluation, visualization, and a reproducibility check.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
print(f'Python: {platform.python_version()}')
print(f'NumPy: {np.__version__}; pandas: {pd.__version__}')

## Generate and validate synthetic data

The fixed random seed makes the generated data and split reproducible.

In [ ]:
X, y = make_regression(
    n_samples=600,
    n_features=5,
    n_informative=4,
    noise=12.0,
    random_state=RANDOM_STATE,
)
feature_names = [f'feature_{i}' for i in range(1, X.shape[1] + 1)]
data = pd.DataFrame(X, columns=feature_names).assign(target=y)

assert data.shape == (600, 6)
assert not data.isna().any().any()
assert data.select_dtypes(include='number').shape[1] == data.shape[1]
data.describe().round(3)

## Train and evaluate

We retain 20% of the rows as an untouched test set and report MAE, RMSE, and R².

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data[feature_names],
    data['target'],
    test_size=0.20,
    random_state=RANDOM_STATE,
)
model = LinearRegression().fit(X_train, y_train)
predictions = model.predict(X_test)

metrics = pd.Series({
    'MAE': mean_absolute_error(y_test, predictions),
    'RMSE': mean_squared_error(y_test, predictions) ** 0.5,
    'R2': r2_score(y_test, predictions),
})
metrics.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, predictions, alpha=0.65)
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
ax.plot(limits, limits, '--', color='black', label='Ideal prediction')
ax.set(xlabel='Actual target', ylabel='Predicted target', title='Actual vs. predicted')
ax.legend()
plt.tight_layout()
plt.show()

## Verify reproducibility

Repeating the split and fit with the same seed must produce identical predictions.

In [ ]:
X_train_2, X_test_2, y_train_2, _ = train_test_split(
    data[feature_names],
    data['target'],
    test_size=0.20,
    random_state=RANDOM_STATE,
)
model_2 = LinearRegression().fit(X_train_2, y_train_2)
predictions_2 = model_2.predict(X_test_2)

np.testing.assert_allclose(predictions, predictions_2)
assert metrics['R2'] > 0.95
print('Reproducibility check passed.')